# SQL in Python - Connecting to and retrieving data from PostgreSQL

Previously, you have learned how to connect to a SQL database by using a SQL client such as DBeaver. Apart from connecting to databases, DBeaver also allows you to run SQL queries against the database, create new tables and populate them with data as well as retrieving the data.

Python also allows executing SQL queries and getting the result into a Python object, for example a Pandas data frame. Instead of exporting a .csv file from DBeaver you can directly get the data you need into Python and continue your work. In addition we can reduce the steps by connecting to the database from Python directly, eliminating the need for a separate SQL client.

After you have the data in Python in the required shape you can export the data into a .csv file. This file is for your own reference, please avoid sending .csv files around - database is the point of reference when it comes to data. 

Having a copy of a .csv file (or another format) can speed up your analysis work. Imagine that the query takes 25 minutes to run. If you made some mistakes in your Python code you might need to go back to the original dataset. Instead of having to rerun the SQL query and having to wait you can read in the .csv file you have previously saved on your hard disk into Python and continue with your analysis work. 

**In this notebook you will see 2 ways to connect to SQL-Databases and export the data to a CSV file**


## Creating a connection to a PostgreSQL database with Python

There are 2 python packages that are the "go-to" when it comes to connecting to SQL-Databases: `psycopg2` and `sqlalchemy` 

### Connecting via psycopg2

In [5]:
%pip install psycopg2-binary


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 3.8 MB 2.6 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import psycopg2


In order to create a connection to our PostgreSQL database we need the following information:

- host = the address of the machine the database is hosted on
- port = the virtual gate number through which communication will be allowed
- database = the name of the database
- user = the name of the user
- password = the password of the user

Because we don't want that the database information is published on GitHub we put it into a `.env` file which is added into the `.gitignore`. 
In these kind of files you can store information that is not supposed to be published.
With the `dotenv` package you can read the `.env` files and get the variables.


In [30]:
import os
from dotenv import load_dotenv

load_dotenv()

DATABASE = os.getenv("DATABASE")
USER_DB = os.getenv("USER_DB")
PASSWORD = os.getenv("PASSWORD")
HOST = os.getenv("HOST")
PORT = os.getenv("PORT")

print(DATABASE)

postgres


The function from the psycopg2 package to create a connection is called `connect()`.
`connect()` expects the parameters listed above as input in order to connect to the database.

In [31]:
# Create connection object conn
conn = psycopg2.connect(
    database=DATABASE, user=USER_DB, password=PASSWORD, host=HOST, port=PORT
)

### Retrieving data from the database with psycopg2

Before we can use our connection to get data, we have to create a cursor. A cursor allows Python code to execute PostgreSQL commands in a database session.
A cursor has to be created with the `cursor()` method of our connection object conn.

In [32]:
cur = conn.cursor()

Now we can run SQL-Queries with `cur.execute('QUERY')` and then run `cur.fetchall()` to get the data:

In [27]:
cur.execute("SELECT * FROM eda.king_county_house_sales LIMIT 10")
rows = cur.fetchall()
for row in rows:
    print(row)

(datetime.date(2014, 10, 13), 221900.0, 7129300520, 1)
(datetime.date(2014, 12, 9), 538000.0, 6414100192, 2)
(datetime.date(2015, 2, 25), 180000.0, 5631500400, 3)
(datetime.date(2014, 12, 9), 604000.0, 2487200875, 4)
(datetime.date(2015, 2, 18), 510000.0, 1954400510, 5)
(datetime.date(2014, 5, 12), 1230000.0, 7237550310, 6)
(datetime.date(2014, 6, 27), 257500.0, 1321400060, 7)
(datetime.date(2015, 1, 15), 291850.0, 2008000270, 8)
(datetime.date(2015, 4, 15), 229500.0, 2414600126, 9)
(datetime.date(2015, 3, 12), 323000.0, 3793500160, 10)


With `conn.close()` you can close the connection again.

In [23]:
# close the connection
conn.close()

But we want to work with the data. The easiest way is to import the data into pandas dataframes. We can use `pd.read_sql_query` or `pd.read_sql_table` or for convenience `pd.read_sql`.

This function is a convenience wrapper around read_sql_table and read_sql_query (for backward compatibility). It will delegate to the specific function depending on the provided input. A SQL query will be routed to read_sql_query , while a database table name will be routed to read_sql_table . Note that the delegated function might have more specific notes about their functionality not listed here.

In [ ]:
# Open connection again because we closed it
conn = psycopg2.connect(
    database=DATABASE, user=USER_DB, password=PASSWORD, host=HOST, port=PORT
)

In [36]:
conn.rollback()

In [42]:
# import the data into a pandas dataframe
query_string = """set schema 'eda';

select kchd.* , kchs.date , kchs.price 
from king_county_house_details as kchd
inner join king_county_house_sales kchs on kchd.id = kchs.house_id ;
"""
df = pd.read_sql(query_string, conn)
df_psycopg = pd.read_sql(query_string, conn)

/var/folders/br/wkt45tkj2bd5stbxtq6czsxm0000gn/T/ipykernel_87597/2338701780.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_string, conn)
/var/folders/br/wkt45tkj2bd5stbxtq6czsxm0000gn/T/ipykernel_87597/2338701780.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_psycopg = pd.read_sql(query_string, conn)


In [ ]:
df.head()

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,date,price
0,7129300520,3.0,1.00,1180.0,5650.0,1.0,NaN,0.0,3,7,...,0.0,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0,2014-10-13,221900.0
1,6414100192,3.0,2.25,2570.0,7242.0,2.0,0.0,0.0,3,7,...,400.0,1951,19910.0,98125,47.7210,-122.319,1690.0,7639.0,2014-12-09,538000.0
2,5631500400,2.0,1.00,770.0,10000.0,1.0,0.0,0.0,3,6,...,0.0,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0,2015-02-25,180000.0
3,2487200875,4.0,3.00,1960.0,5000.0,1.0,0.0,0.0,5,7,...,910.0,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0,2014-12-09,604000.0
4,1954400510,3.0,2.00,1680.0,8080.0,1.0,0.0,0.0,3,8,...,0.0,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0,2015-02-18,510000.0


In [41]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21597 entries, 0 to 21596
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21597 non-null  int64  
 1   bedrooms       21597 non-null  float64
 2   bathrooms      21597 non-null  float64
 3   sqft_living    21597 non-null  float64
 4   sqft_lot       21597 non-null  float64
 5   floors         21597 non-null  float64
 6   waterfront     19206 non-null  float64
 7   view           21534 non-null  float64
 8   condition      21597 non-null  int64  
 9   grade          21597 non-null  int64  
 10  sqft_above     21597 non-null  float64
 11  sqft_basement  21145 non-null  float64
 12  yr_built       21597 non-null  int64  
 13  yr_renovated   17749 non-null  float64
 14  zipcode        21597 non-null  int64  
 15  lat            21597 non-null  float64
 16  long           21597 non-null  float64
 17  sqft_living15  21597 non-null  float64
 18  sqft_lot15     21

In [68]:
df.describe()

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,price
count,2.159700e+04,21597.000000,21597.000000,21597.000000,2.159700e+04,21597.000000,19206.000000,21534.000000,21597.000000,21597.000000,21597.000000,21145.000000,21597.000000,17749.000000,21597.000000,21597.000000,21597.000000,21597.000000,21597.000000,2.159700e+04
mean,4.580474e+09,3.373200,2.115826,2080.321850,1.509941e+04,1.494096,0.007602,0.233863,3.409825,7.657915,1788.596842,291.857224,1970.999676,836.650516,98077.951845,47.560093,-122.213983,1986.620318,12758.283512,5.402966e+05
std,2.876736e+09,0.926299,0.768984,918.106125,4.141264e+04,0.539683,0.086858,0.765686,0.650546,1.173200,827.759761,442.490863,29.375234,4000.110554,53.513072,0.138552,0.140724,685.230472,27274.441950,3.673681e+05
min,1.000102e+06,1.000000,0.500000,370.000000,5.200000e+02,1.000000,0.000000,0.000000,1.000000,3.000000,370.000000,0.000000,1900.000000,0.000000,98001.000000,47.155900,-122.519000,399.000000,651.000000,7.800000e+04
25%,2.123049e+09,3.000000,1.750000,1430.000000,5.040000e+03,1.000000,0.000000,0.000000,3.000000,7.000000,1190.000000,0.000000,1951.000000,0.000000,98033.000000,47.471100,-122.328000,1490.000000,5100.000000,3.220000e+05
50%,3.904930e+09,3.000000,2.250000,1910.000000,7.618000e+03,1.500000,0.000000,0.000000,3.000000,7.000000,1560.000000,0.000000,1975.000000,0.000000,98065.000000,47.571800,-122.231000,1840.000000,7620.000000,4.500000e+05
75%,7.308900e+09,4.000000,2.500000,2550.000000,1.068500e+04,2.000000,0.000000,0.000000,4.000000,8.000000,2210.000000,560.000000,1997.000000,0.000000,98118.000000,47.678000,-122.125000,2360.000000,10083.000000,6.450000e+05
max,9.900000e+09,33.000000,8.000000,13540.000000,1.651359e+06,3.500000,1.000000,4.000000,5.000000,13.000000,9410.000000,4820.000000,2015.000000,20150.000000,98199.000000,47.777600,-121.315000,6210.000000,871200.000000,7.700000e+06


In [44]:
df.isnull()

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,date,price
0,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21592,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
21593,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
21594,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
21595,False,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [76]:
nummeric_columns = df.select_dtypes(include="number").columns
categorical_columns = df.select_dtypes(include=["object", "category", "bool"]).columns
print(nummeric_columns.value_counts().sum())
print(categorical_columns.value_counts().sum())

20
1


In [85]:
print("Numeric columns:")
print(nummeric_columns.tolist())


Numeric columns:
['id', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15', 'price']


In [ ]:
print("Categorical columns:")
print(categorical_columns.tolist())


Categorical columns:
['date']


In [51]:
column_summary = pd.DataFrame({
    "dtype": df_psycopg.dtypes,
    "unique_values": df_psycopg.nunique(),
    "missing_values": df_psycopg.isna().sum()
})

column_summary

,dtype,unique_values,missing_values
id,int64,21420,0
bedrooms,float64,12,0
bathrooms,float64,29,0
sqft_living,float64,1034,0
sqft_lot,float64,9776,0
floors,float64,6,0
waterfront,float64,2,2391
view,float64,5,63
condition,int64,5,0
grade,int64,11,0


In [49]:
df.nunique().sort_values()

waterfront           2
view                 5
condition            5
floors               6
grade               11
bedrooms            12
bathrooms           29
yr_renovated        70
zipcode             70
yr_built           116
sqft_basement      303
date               372
long               752
sqft_living15      777
sqft_above         942
sqft_living       1034
price             3622
lat               5033
sqft_lot15        8682
sqft_lot          9776
id               21420
dtype: int64

In [50]:
df.dtypes

id                 int64
bedrooms         float64
bathrooms        float64
sqft_living      float64
sqft_lot         float64
floors           float64
waterfront       float64
view             float64
condition          int64
grade              int64
sqft_above       float64
sqft_basement    float64
yr_built           int64
yr_renovated     float64
zipcode            int64
lat              float64
long             float64
sqft_living15    float64
sqft_lot15       float64
date              object
price            float64
dtype: object

In [52]:
df.columns.tolist()

['id',
 'bedrooms',
 'bathrooms',
 'sqft_living',
 'sqft_lot',
 'floors',
 'waterfront',
 'view',
 'condition',
 'grade',
 'sqft_above',
 'sqft_basement',
 'yr_built',
 'yr_renovated',
 'zipcode',
 'lat',
 'long',
 'sqft_living15',
 'sqft_lot15',
 'date',
 'price']

In [53]:
print(f'rows: {df.shape[0]}')
print(f'columns: {df.shape[1]}')

rows: 21597
columns: 21


In [54]:
df.shape

(21597, 21)

In [58]:
df.columns[df.columns.duplicated()].tolist()

[]

In [59]:
df.duplicated().sum()

np.int64(0)

In [60]:
os.getcwd()

'/Users/elhamrouzban/neue_fische/screen-shots/EDA_Presenting_your_Results/ds-eda-project-template'

In [61]:
df.to_csv("data/king_county_housing.csv", index=False, encoding="utf-8")
os.path.exists("data/king_county_housing.csv")

True

In [62]:
df_csv = pd.read_csv("data/king_county_housing.csv")
df_csv.head()

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,date,price
0,7129300520,3.0,1.00,1180.0,5650.0,1.0,NaN,0.0,3,7,...,0.0,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0,2014-10-13,221900.0
1,6414100192,3.0,2.25,2570.0,7242.0,2.0,0.0,0.0,3,7,...,400.0,1951,19910.0,98125,47.7210,-122.319,1690.0,7639.0,2014-12-09,538000.0
2,5631500400,2.0,1.00,770.0,10000.0,1.0,0.0,0.0,3,6,...,0.0,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0,2015-02-25,180000.0
3,2487200875,4.0,3.00,1960.0,5000.0,1.0,0.0,0.0,5,7,...,910.0,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0,2014-12-09,604000.0
4,1954400510,3.0,2.00,1680.0,8080.0,1.0,0.0,0.0,3,8,...,0.0,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0,2015-02-18,510000.0


In [66]:
print("Original:", df.shape)
print("CSV:", df_csv.shape)

Original: (21597, 21)
CSV: (21597, 21)


In [67]:
df_csv.info()

<class 'pandas.DataFrame'>
RangeIndex: 21597 entries, 0 to 21596
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21597 non-null  int64  
 1   bedrooms       21597 non-null  float64
 2   bathrooms      21597 non-null  float64
 3   sqft_living    21597 non-null  float64
 4   sqft_lot       21597 non-null  float64
 5   floors         21597 non-null  float64
 6   waterfront     19206 non-null  float64
 7   view           21534 non-null  float64
 8   condition      21597 non-null  int64  
 9   grade          21597 non-null  int64  
 10  sqft_above     21597 non-null  float64
 11  sqft_basement  21145 non-null  float64
 12  yr_built       21597 non-null  int64  
 13  yr_renovated   17749 non-null  float64
 14  zipcode        21597 non-null  int64  
 15  lat            21597 non-null  float64
 16  long           21597 non-null  float64
 17  sqft_living15  21597 non-null  float64
 18  sqft_lot15     21

Understanding the Data

The dataset contains 21,597 observations (rows) and 21 variables (columns)

The dataset includes the following variables:

Column	Description   &  Data Types
- id          int64
- bedrooms     float64
- bathrooms   float64
- sqft_living   float64
- sqft_lot     float64
- floors       float64
- waterfront   float64
- view         float64
- condition    int64
- grade        int64
- sqft_above   float64
- sqft_basement float64
- yr_built      int64
- yr_renovated  float64
- zipcode       int64
- lat          float64
- long         float64
- sqft_living15  float64
- sqft_lot15     float64
- date          str
- price         float64


The dataset consists of:

15 floating-point (float64) variables
5 integer (int64) variables
1 string (str) variable

The date column is currently stored as a string. Since it represents dates rather than text, it should later be converted to a datetime data type to enable time-based analysis.

Missing Values

| Column            | Non-Null Values | Missing Values |
| ----------------- | --------------: | -------------: |
| id                |          21,597 |              0 |
| bedrooms          |          21,597 |              0 |
| bathrooms         |          21,597 |              0 |
| sqft_living       |          21,597 |              0 |
| sqft_lot          |          21,597 |              0 |
| floors            |          21,597 |              0 |
| **waterfront**    |      **19,206** |      **2,391** |
| **view**          |      **21,534** |         **63** |
| condition         |          21,597 |              0 |
| grade             |          21,597 |              0 |
| sqft_above        |          21,597 |              0 |
| **sqft_basement** |      **21,145** |        **452** |
| yr_built          |          21,597 |              0 |
| **yr_renovated**  |      **17,749** |      **3,848** |
| zipcode           |          21,597 |              0 |
| lat               |          21,597 |              0 |
| long              |          21,597 |              0 |
| sqft_living15     |          21,597 |              0 |
| sqft_lot15        |          21,597 |              0 |
| date              |          21,597 |              0 |
| price             |          21,597 |              0 |


These four variables contain missing values: 
waterfront, view, sqft_basement, and yr_renovated. 
The remaining seventeen variables have no missing observations.


In [ ]:
# close the connection
conn.close()

In [ ]:
df_psycopg.head()

In [ ]:
# export the data to a csv-file
df_psycopg.to_csv("data/eda.csv", index=False)

### Connecting and retrieving data via SQLAlchemy

`sqlalchemy` works similarly. Here you have to create an engine with the database string (a link that includes every information we entered in the conn object)

In [ ]:
from sqlalchemy import create_engine

# read the database string from the .env
load_dotenv()

DB_STRING = os.getenv("DB_STRING")

if DB_STRING is None:
    raise ValueError("DB_STRING is not set in the environment.")

db = create_engine(DB_STRING)

And then you can import that engine with a query into a pandas dataframe.

In [ ]:
# import the data to a pandas dataframe
query_string = "SELECT * FROM eda.king_county_house_sales"
df_sqlalchemy = pd.read_sql(query_string, db)

In [ ]:
df_sqlalchemy.head()

Because we don't want to run the queries over and over again we can export the data into a .csv file in order to use it in other notebooks as well. 

In [ ]:
# export the data to a csv-file
df_sqlalchemy.to_csv("data/eda.csv", index=False)

In [ ]:
# import the data from a csv-file
df_import = pd.read_csv("data/eda.csv")